# 🚀 CSIRO Image2Biomass: Dual-Stream Inference & Submission
### Multi-Fold DINO Ensembling with Test-Time Augmentation (TTA) & Soft Physical Post-Processing

This notebook executes the complete inference pipeline on `test.csv`:
1. **Discovers all 5-Fold Trained Checkpoints** (`models/best_model_fold*.pt`).
2. **Dual-Stream 1:1 Square Splitting**: splits 2000×1000 panoramic test images into Left and Right views.
3. **Test-Time Augmentation (TTA)**: Evaluates normal views and horizontally flipped mirrored views.
4. **Soft Physical Blend Post-Processing**: Blends direct Total/GDM predictions with physical mass sums, adjusts Dead extremes, and scales Clover ($0.8\times$).
5. **Competition Submission Formatting**: Outputs `submission.csv` strictly formatted as `sample_id,target`.

In [ ]:
# 1. Environment & Setup
import os
import sys
import glob
import numpy as np
import pandas as pd
from PIL import Image
import cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

TARGET_NAMES = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

## 2. Test Dataset & Model Definition

In [ ]:
class DualStreamTestDataset(Dataset):
    def __init__(self, df, img_dir=None, img_size=512):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, rel_path):
        if self.img_dir:
            fname = os.path.basename(rel_path)
            cand = os.path.join(self.img_dir, fname)
            if os.path.exists(cand):
                return cand
        if os.path.exists(rel_path):
            return rel_path
        fname = os.path.basename(rel_path)
        for cand_dir in ['test', 'train', 'images', os.path.join('..', 'test'), os.path.join('..', 'train')]:
            cand = os.path.join(cand_dir, fname)
            if os.path.exists(cand):
                return cand
        raise FileNotFoundError(f"Image not found: {rel_path}")

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_image_path(row['image_path'])
        raw_bgr = cv2.imread(img_path)
        raw_rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
        h, w, _ = raw_rgb.shape
        mid_w = w // 2
        
        left_np = raw_rgb[:, :mid_w].copy()
        right_np = raw_rgb[:, mid_w:].copy()
        
        tensor_l = self.transform(Image.fromarray(left_np))
        tensor_r = self.transform(Image.fromarray(right_np))
        
        return {
            'image_left': tensor_l,
            'image_right': tensor_r,
            'image_path': img_path,
            'clean_id': os.path.splitext(os.path.basename(img_path))[0],
            'state': row.get('State', 'Unknown')
        }

class DualStreamDINO(nn.Module):
    def __init__(self, backbone_name="vit_base_patch14_dinov2", fusion_dim=384, dropout=0.3, num_targets=5):
        super().__init__()
        self.backbone_name = backbone_name
        self.fusion_dim = fusion_dim
        self.num_targets = num_targets
        
        kwargs = {}
        if 'dinov2' in backbone_name or 'patch14' in backbone_name or 'patch16' in backbone_name:
            kwargs['dynamic_img_size'] = True
            
        self.backbone = timm.create_model(backbone_name, pretrained=False, num_classes=0, **kwargs)
        self.backbone_dim = self.backbone.num_features
        
        n_heads = 8 if self.backbone_dim % 8 == 0 else 4
        self.cross_view_attn = nn.MultiheadAttention(
            embed_dim=self.backbone_dim, num_heads=n_heads, dropout=0.1, batch_first=True
        )
        self.attn_norm = nn.LayerNorm(self.backbone_dim)
        
        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, self.fusion_dim),
            nn.LayerNorm(self.fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.reg_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.fusion_dim, self.fusion_dim // 2),
                nn.LayerNorm(self.fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(self.fusion_dim // 2, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            ) for _ in range(self.num_targets)
        ])
        
        self.cls_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(self.fusion_dim, 128),
                nn.LayerNorm(128),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(128, 7)
            ) for _ in range(self.num_targets)
        ])

    def extract_features(self, x):
        feats = self.backbone(x)
        if len(feats.shape) == 3:
            return feats.mean(dim=1)
        elif len(feats.shape) == 4:
            return feats.mean(dim=[2, 3])
        return feats

    def forward(self, img_left, img_right):
        feat_l = self.extract_features(img_left)
        feat_r = self.extract_features(img_right)
        tokens = torch.stack([feat_l, feat_r], dim=1)
        attn_out, _ = self.cross_view_attn(tokens, tokens, tokens)
        tokens = self.attn_norm(tokens + attn_out)
        fused = torch.cat([tokens[:, 0], tokens[:, 1]], dim=-1)
        fused = self.fusion_mlp(fused)
        reg_preds = [F.softplus(head(fused)) for head in self.reg_heads]
        cls_preds = [head(fused) for head in self.cls_heads]
        return reg_preds, cls_preds

print("✓ Test dataset and DualStreamDINO model classes defined.")

## 3. Discover Fold Checkpoints & Run Multi-Fold TTA Inference

In [ ]:
def auto_detect_backbone(state_dict):
    clean_sd = {k.replace('module.', ''): v for k, v in state_dict.items()}
    if 'cross_view_attn.in_proj_weight' in clean_sd:
        dim = clean_sd['cross_view_attn.in_proj_weight'].shape[1]
        if dim == 768: return 'vit_base_patch14_dinov2'
        elif dim == 384: return 'vit_small_patch14_dinov2'
        elif dim == 1024: return 'vit_large_patch14_dinov2'
        elif dim == 1536: return 'convnextv2_large'
    return 'vit_base_patch14_dinov2'

# 1. Find Checkpoints
ckpt_paths = sorted(glob.glob("models/best_model_fold*.pt"))
if not ckpt_paths:
    ckpt_paths = sorted(glob.glob("logs/**/best_model_fold*.pt", recursive=True))
print(f"Found {len(ckpt_paths)} checkpoint(s): {[os.path.basename(p) for p in ckpt_paths]}")

# 2. Load Test Metadata
test_csv_path = 'test.csv'
test_df_raw = pd.read_csv(test_csv_path)
unique_test_df = test_df_raw.drop_duplicates(subset=['image_path']).reset_index(drop=True)
print(f"Loaded {len(test_df_raw)} test rows across {len(unique_test_df)} unique images.")

test_loader = DataLoader(
    DualStreamTestDataset(unique_test_df, img_size=512),
    batch_size=8, shuffle=False, num_workers=2
)

# 3. Run Inference Across All Fold Models with TTA
all_fold_preds = []
for ckpt_idx, ckpt_path in enumerate(ckpt_paths):
    print(f"Predicting with Fold {ckpt_idx + 1}/{len(ckpt_paths)}: {os.path.basename(ckpt_path)}...")
    sd = torch.load(ckpt_path, map_location=device, weights_only=True)
    backbone = auto_detect_backbone(sd)
    
    head_indices = [int(k.split('.')[1]) for k in sd.keys() if k.startswith('reg_heads.') and '.0.weight' in k]
    num_targets = max(head_indices) + 1 if head_indices else 5
    
    model = DualStreamDINO(backbone_name=backbone, num_targets=num_targets).to(device)
    model.load_state_dict(sd, strict=False)
    model.eval()
    
    fold_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, leave=False):
            img_l = batch['image_left'].to(device)
            img_r = batch['image_right'].to(device)
            # TTA: Standard + Horizontally Flipped
            r1, _ = model(img_l, img_r)
            r2, _ = model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
            reg_preds = [(a + b) * 0.5 for a, b in zip(r1, r2)]
            p_matrix = torch.cat(reg_preds, dim=1).cpu().numpy()
            fold_preds.append(p_matrix)
            
    all_fold_preds.append(np.concatenate(fold_preds, axis=0))

ensemble_raw = np.mean(all_fold_preds, axis=0)
print(f"✓ Ensembled {len(ckpt_paths)} models successfully.")

## 4. Soft Physical Blend Post-Processing
Applies the 1st/2nd/3rd place winning post-processing:
1. Clover multiplier ($0.8\times$)
2. Dead extremes scaling ($>20\times 1.1, <10\times 0.9$)
3. Blends direct Total/GDM predictions with physical mass sums
4. Non-negative clipping

In [ ]:
def apply_soft_blend_postprocess(preds_5, states=None):
    preds = np.maximum(np.asarray(preds_5, dtype=np.float32).copy(), 0.0)
    green = preds[:, 0]
    dead = preds[:, 1]
    clover = preds[:, 2] * 0.8
    gdm = preds[:, 3]
    total = preds[:, 4]
    
    dead = np.where(dead > 20.0, dead * 1.1, np.where(dead < 10.0, dead * 0.9, dead))
    if states is not None:
        for idx, st in enumerate(states):
            if str(st).strip() == 'WA':
                dead[idx] = 0.0
                
    gdm_blended = 0.5 * gdm + 0.5 * (green + clover)
    total_blended = 0.5 * total + 0.5 * (green + clover + dead)
    return np.maximum(np.column_stack([green, dead, clover, gdm_blended, total_blended]), 0.0)

states = unique_test_df['State'].tolist() if 'State' in unique_test_df.columns else None
ensemble_post = apply_soft_blend_postprocess(ensemble_raw, states=states)
print("✓ Soft physical post-processing applied.")

## 5. Generate & Verify `submission.csv`

In [ ]:
img_to_preds = {}
for idx, row in unique_test_df.iterrows():
    img_to_preds[row['image_path']] = ensemble_post[idx]

submission_rows = []
for _, row in test_df_raw.iterrows():
    img_p = row['image_path']
    t_name = row['target_name']
    s_id = row['sample_id']
    
    if img_p in img_to_preds:
        t_idx = TARGET_NAMES.index(t_name)
        val = float(img_to_preds[img_p][t_idx])
    else:
        val = 0.0
        
    submission_rows.append({'sample_id': s_id, 'target': val})

sub_df = pd.DataFrame(submission_rows)
sub_df.to_csv('submission.csv', index=False)
print(f"🎉 Created submission.csv with {len(sub_df)} rows.")
display(sub_df.head(10))

print("\nSummary Statistics by Target:")
for t in TARGET_NAMES:
    sub_t = sub_df[sub_df['sample_id'].str.contains(t)]
    if len(sub_t) > 0:
        print(f"  {t:15s}: mean={sub_t['target'].mean():.2f} min={sub_t['target'].min():.2f} max={sub_t['target'].max():.2f}")